In [389]:
import glob
import numpy as np
import pandas as pd

CSV_GLOB = "/project2/ll_774_951/midterm/*/*.csv"

REP_HASHTAGS = [
    # "maga",
    "voteredtosaveamerica",
    "votered",
    "redwavecoming",
    "democratsaretheproblem",
    # new
    "2a",
    "1a",
    "fjb",
    "americafirst",
    "kag"
]
DEM_HASHTAGS = [
    "voteblue",
    "voteblue2022",
    "votebluetosavedemocracy",
    "votebluetosaveamerica",
    "votebluein2022",
    "votebluenomatterwho",
    "votebluefordemocracy",
    "votebluetoprotectwomen",
    "voteblueforwomensrights",
    "votebluetoprotectyourrights",
    "voteblueforsomanyreasons",
    "votebluetoendtheinsanity",
    "votebluenotq",
    "votebluedownballot",
    "votebluedownballotlocalstatefederal",
    "votebluetosavesocialsecurity",
    "votebluetosavesocialsecurityandmedicare",
    "votebluetosaveourkids",
    "bluewave",
    "bluewave2022",
    "bluecrew",
    "bluevoters",
    "ourbluevoice",
    "bluein22",
    "proudblue22",
    "demvoice1",
    "wtpblue",
    "democratsdeliver",
    "demsact",
    "voteouteveryrepublican",
    "stopvotingforrepublicans",
    "neverrepublicanagain",
    "republicansaretheproblem",
    "republicanwaronwomen",
    "goptraitorstodemocracy",
    "gopliesabouteverything",
    "magaidiots",
    # new
    "blm",
    "blacklivesmatter",
    "resist",
    "fbr"
]
DEM_MEDIA_OUTLETS = [
    "abcnews", "bbc", "buzzfeednews", "huffpost", "msnbc", "cnn",
    "nytimes", "washingtonpost", "latimes", "guardian",
]
REP_MEDIA_OUTLETS = [
    "breitbartnews", "dailycaller", "dailymail", "foxnews", "infowars", "oann", 
    # new
    "breitbart"
]

In [2]:
usecols = ["userid", "date", "description", "urls", "urls_list"]
dfs = []
for fp in sorted(glob.glob(CSV_GLOB)):
    try:
        dfi = pd.read_csv(fp, low_memory=False, on_bad_lines="skip", usecols=lambda c: c in usecols)
        if len(dfi):
            dfs.append(dfi)
    except Exception as e:
        print("skip", fp, e)

df = pd.concat(dfs, ignore_index=True)
df["userid"] = pd.to_numeric(df["userid"], errors="coerce")
df = df.dropna(subset=["userid"]).copy()
df["userid"] = df["userid"].astype(np.int64)

df["date"] = pd.to_datetime(
    df["date"].astype(str).str.strip(),
    format="%a %b %d %H:%M:%S +0000 %Y",
    utc=True,
    errors="coerce",
)
df = df.dropna(subset=["date"]).copy()

df["description"] = df.get("description", "").fillna("").astype(str).str.lower()

def f(x):
    try:
        return eval(x)
    except:
        print(x)
        return []
df['urls_raw'] = df.urls_list.fillna('[]').apply(f)

skip /project2/ll_774_951/midterm/converted_data_2022-10-05-00_2022-10-05-23/midterm-2022-10-05-19.csv Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.

skip /project2/ll_774_951/midterm/converted_data_2022-10-07-00_2022-10-07-23/midterm-2022-10-07-15.csv Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.

skip /project2/ll_774_951/midterm/converted_data_2022-10-10-00_2022-10-10-23/midterm-2022-10-10-09.csv Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.

skip /project2/ll_774_951/midterm/converted_data_2022-10-10-00_2022-10-10-23/midterm-2022-10-10-13.csv Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.

skip /project2/ll_774_951/midterm/converted_data_2022-10-10-00_2022-10-10-23/midterm-2022-10-10-16.csv Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.

skip /project2/ll_774_951/midterm/conver

In [46]:
df['urls'] = df[df['urls_raw'].str.len().gt(0)].urls.apply(lambda x: [xx['expanded_url'] for xx in x])

In [390]:
rep_media_pat = "|".join(REP_MEDIA_OUTLETS)
dem_media_pat = "|".join(DEM_MEDIA_OUTLETS)

df['rep_domains'] = df.urls.explode().str.findall(rep_media_pat).explode().dropna().groupby(level=0).agg(list)
user2rep_dom = df.explode('rep_domains').dropna(subset=['rep_domains']).groupby('userid')['rep_domains'].agg(list) 

df['has_rep_dom'] = df.rep_domains.str.len()
user2rep_dom_tweet_count = df.groupby('userid').has_rep_dom.sum()

df['dem_domains'] = df.urls.explode().str.findall(dem_media_pat).explode().dropna().groupby(level=0).agg(list)
user2dem_dom = df.explode('dem_domains').dropna(subset=['dem_domains']).groupby('userid')['dem_domains'].agg(list)

df['has_dem_dom'] = df.dem_domains.str.len()
user2dem_dom_tweet_count = df.groupby('userid').has_dem_dom.sum()

In [423]:
rep_hash_pat = r"\#(?:" + "|".join(REP_HASHTAGS) + r")"
dem_hash_pat = r"\#(?:" + "|".join(DEM_HASHTAGS) + r")"

# g = df.drop_duplicates(['userid','description']).copy()
df['rep_hashs'] = df.description.str.findall(rep_hash_pat)
df['rep_hashs_len'] = df['rep_hashs'].str.len()

user2rep_hash = (
    df.sort_values('rep_hashs_len')
        .drop_duplicates(['userid','description'], keep='first')
        .groupby('userid')
        .agg({'rep_hashs': 'sum'})
)
user2rep_hash['rep_hashs'] = user2rep_hash['rep_hashs'].apply(set)
user2rep_hash['rep_hashs_len'] = user2rep_hash.rep_hashs.str.len()

df['dem_hashs'] = df.description.str.findall(dem_hash_pat)
df['dem_hashs_len'] = df['dem_hashs'].str.len()

user2dem_hash = (
    df.sort_values('dem_hashs_len')
        .drop_duplicates(['userid','description'], keep='first')
        .groupby('userid')
        .agg({'dem_hashs': 'sum'})
)
user2dem_hash['dem_hashs'] = user2dem_hash['dem_hashs'].apply(set)
user2dem_hash['dem_hashs_len'] = user2dem_hash.dem_hashs.str.len()

In [431]:
user2pol = user2rep_hash.join(user2dem_hash)
user2pol['user2rep_dom_tweet_count'] = user2rep_dom_tweet_count
user2pol['user2dem_dom_tweet_count'] = user2dem_dom_tweet_count

In [ ]:
is_rep = (
    (user2pol.user2rep_dom_tweet_count.gt(0) | user2pol.rep_hashs_len.gt(0))
    # (counts.rep_dom_tweet_count.gt(0) & counts.rep_hash_count.gt(0))
)

is_dem = (
    (user2pol.user2dem_dom_tweet_count.gt(0) | user2pol.dem_hashs_len.gt(0))
    # (counts.dem_dom_tweet_count.gt(0) & counts.dem_hash_count.gt(0))
)

print((is_rep.sum(), is_dem.sum()), (is_rep.sum() + is_dem.sum()))

user2pol['is_rep'] = is_rep
user2pol['is_dem'] = is_dem
user2pol[['is_rep', 'is_dem']].sum(1).gt(1).mean(), user2pol[['is_rep', 'is_dem']].sum(1).gt(1).sum()

(6631, 28100) 34731


(0.008791281233887163, 3734)

In [ ]:
df['user2is_rep'] = df.userid.map(user2pol['is_rep'])
df['user2is_dem'] = df.userid.map(user2pol['is_dem'])
df['user2rep_dom_tweet_count'] = df.userid.map(user2pol.user2rep_dom_tweet_count)
df['user2dem_dom_tweet_count'] = df.userid.map(user2pol.user2dem_dom_tweet_count)
df['user2rep_hash_count'] = df.userid.map(user2pol.rep_hashs_len)
df['user2dem_hash_count'] = df.userid.map(user2pol.dem_hashs_len)

In [ ]:
user2pol = user2pol[~(user2pol.is_rep & user2pol.is_dem)]

,rep_hashs,rep_hashs_len,dem_hashs,dem_hashs_len,user2rep_dom_tweet_count,user2dem_dom_tweet_count,is_rep,is_dem
userid,,,,,,,,
5405752,{},0,{},0,1.0,0.0,True,True
5693842,{},0,{},0,1.0,0.0,True,True
6021962,{},0,{},0,1.0,0.0,True,True
6124022,{},0,{},0,1.0,0.0,True,True
7871422,{},0,{},0,1.0,0.0,True,True
...,...,...,...,...,...,...,...,...
1582450373201170432,{},0,{},0,1.0,0.0,True,True
1582697195354161152,{},0,{},0,1.0,0.0,True,True
1582926521303707648,{},0,{},0,1.0,0.0,True,True
